# 스스로 판단하고 행동하는 AI 요원, 에이전트

### 1.1 가장 단순한 도구(함수) 만들기

### 필요 패키지 설치
- 아래 코드 셀을 실행하여 필요한 패키지를 설치합니다.

In [37]:
# %pip install -U langchain langchain-core langchain-openai langchain-community

In [38]:
%pip show langchain

Name: langchain
Version: 1.3.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\workspaces\ai_agent\src\.venv\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


#### 도구 정의(펑션 콜링 컨셉)
- 단순한 형태의 함수를 하나 정의하고, `tool` 데코레이터 부착
- docstring을 추가
- 도구 등록이 간단  : from langchain_core.tools import tool

In [39]:
from langchain_core.tools import tool

# 데코레이터 부착(함수명, 역할, 입력 파라미터.. 등 정의)  gpt_functions.py tools = [{}]
@tool
def greet(name):
    """
    사용자의 이름을 받아 인사합니다.
    """
    return f'안녕, {name}!!!'

print(greet.invoke({'name': '철수'}))

안녕, 철수!!!


### 1.2 도구 메타데이터 다듬기
#### 도구에 상세한 이름 부여하기
- `tool` 데코레이터에 이름 인자 전달

In [40]:
from langchain_core.tools import tool


# 펑션 콜링
# 도구 함수로 등록
@tool('math_calculator')
def calculator(expression: str) -> str:
    """수학 계산이 필요할 때만 사용하세요."""
    return str(eval(expression))  # eval함수 : 입력 값을 그대로 수식으로 반환

print(calculator.name)
print(calculator.description)
print(calculator.invoke({'expression': '2 + 3 * 4'}))


math_calculator
수학 계산이 필요할 때만 사용하세요.
14


#### 도구에 설명 추가하기
- tool 데코레이터의 description 인자로 도구 설명 전달

In [41]:
@tool('math_calculator', description='표현식(문자열)을 계산하여 반환')
def calculator(expression):
# def calculator(expression: str) -> str:
    """
    간단한 수학 표현식을 계산합니다.
    Args:
        expression (str): 계산할 수학 표현식(예: "2 + 3 * 4")
    Returns:
        str: 계산 결과
    """
    return str(eval(expression))

print(calculator.name)
print(calculator.description)
print(calculator.invoke({'expression': '2 + 3 * 4'}))

math_calculator
표현식(문자열)을 계산하여 반환
14


### 1.3 Pydantic 스키마로 구조화된 입력 받기
#### Pydantic 라이브러리
- BaseModel을 상속한 클래스로 데이터의 구조schema를 정의

In [42]:
from pydantic import BaseModel

class SearchInput(BaseModel): # 두개 필드만 정의
    query: str
    top_k: int = 5


#### 도구에서 구조화된 데이터 입력받기
- 도구에서 입력 받을 데이터를 Pydantic으로 구조화시켜 받을 수 있도록 합니다.

In [43]:
from pydantic import BaseModel, Field

# 데이터에 대한 정합성을 가질 수 있음
class WeatherArgs(BaseModel):
    city: str = Field(description='도시이름')
    units: str = Field(description='섭씨 또는 화씨', default='썹씨')

@tool(args_schema=WeatherArgs)
def get_weather_units(city, units="썹씨"):
   """해당 도시에서 주로 사용하는 온도 단위를 반환합니다."""
   return f'{city}(은/는) 주로 {units} 단위를 사용합니다.'

# 도구 함수
print(get_weather_units.invoke({'city': '서울'}))  # units : default='썹씨'


서울(은/는) 주로 썹씨 단위를 사용합니다.


## 2. 에이전트 첫걸음
### 2.1 단일 에이전트 생성하기

In [44]:
from dotenv import load_dotenv

load_dotenv()

True

####	모델 지정과 에이전트 생성

In [45]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# LLM 준비
model = ChatOpenAI(model="gpt-5-nano")

# Agent 생성
agent = create_agent(
    model='gpt-5-nano',
    system_prompt='당신은 도움이 되는 AI 어시스턴트 입니다.'
)


####	에이전트 실행하기

In [46]:
result = agent.invoke({
    'messages': [{'role': 'user', 'content': '5 *12 는?'}]
})

print(result['messages'][-1].content)

5 × 12는 60입니다.


### 2.2 도구를 활용하는 에이전트 생성하기
####	도구를 포함한 에이전트 생성하기

In [47]:
agent = create_agent(
    model='gpt-5-nano',
    tools=[calculator, get_weather_units],  # 도구 함수의 정보를 전달
    system_prompt="""
        당신은 도움이 되는 AI 어시스턴트입니다. 사용자의 질문에 정확하게 답하세요.
        상황에 맞는 도구를 반드시 사용하여 답변합니다.
    """
)

####	에이전트 실행하기
- 수학 계산 질문

In [48]:
result = agent.invoke({
    # "messages": [{"role": "user", "content": "5 × 12는?"}]
    "messages": [{"role": "user", "content": "5 * 12는?"}]
})

print(result["messages"][-1].content)

60입니다. (5 × 12 = 60)


- 미국의 온도 단위를 묻는 질문

In [49]:
result = agent.invoke({
    "messages": [{'role': 'user', 'content': '미국은 어떤 온도 단위를 주로 사용하나요?'}]   
})
print(result['messages'][-1].content)

미국은 주로 화씨(Fahrenheit) 단위를 사용합니다. 다만 과학, 의학, 기상 데이터 등 일부 분야에서는 섭씨(Celsius)도 사용될 수 있습니다. 필요하면 화씨-섭씨 간 변환도 도와드릴게요. 예를 들어 72°F는 약 22.2°C입니다.


#### 랭체인이란 무엇인가를 묻는 질문

In [50]:
result = agent.invoke({
    "messages": [{'role': 'user', 'content': '랭체인이란 무엇인가요?'}]   
})
print(result['messages'][-1].content)

랭체인(LangChain)은 언어나 문장을 다루는 대형 언어 모델(LLM)을 활용한 애플리케이션을 쉽게 만들 수 있게 도와주는 프레임워크입니다. 모델 호출과 다양한 도구를 연결해 “체인(chains)”이나 “에이전트(agents)” 방식으로 동작하도록 구성할 수 있습니다. 즉, 단순한 챗봇을 넘어서 데이터 검색, 문서 요약, 코드 생성, 자동화된 작업 등 복잡한 워크플로우를 손쉽게 구현할 수 있게 해주는 도구 모음이라고 보면 됩니다.

주요 아이디어와 구성 요소
- 체인(Chains): 여러 단계의 언어 모델 호출과 기타 작업을 순차적 또는 조건부로 엮은 연쇄 작업 흐름. 예를 들어, 먼저 요약하고, 그 요약을 바탕으로 질문에 답하는 식으로 구성 가능.
- 프롬프트 템플릿(Prompt Templates): 입력 변수를 받아 LLM에 전달할 프롬프트를 미리 정의하는 방식. 재사용성과 일관성 확보에 도움.
- LLM 래퍼(LLM Wrappers): 다양한 LLM 제공자(OpenAI, Cohere, Hugging Face 등)의 API를 추상화해 같은 인터페이스로 다룰 수 있게 해줌.
- 에이전트(Agents)와 도구(Tools): 에이전트는 상황에 따라 목표를 달성하기 위해 외부 도구(웹 검색, 계산기, 파일 시스템, API 호출 등)를 호출하며 스스로 의사결정을 내림.
- 메모리(Memory): 대화 맥락이나 상태를 유지해 연속적인 대화나 작업 흐름에서 정보를 기억하고 활용.
- 데이터 로더와 벡터 저장소(Vector Stores): 문서를 불러와 임베딩하고, 유사도 검색으로 관련 정보를 빠르게 찾아내는 기능. 예: FAISS, Pinecone, Weaviate 같은 벡터DB와 통합.
- 데이터 소스와 인덱싱: 문서, 데이터베이스, API 등을 연결해 정보에 대한 인덱싱과 검색을 가능하게 함.

자주 쓰이는 용도
- 기억을 갖춘 대화형 에이전트: 과거 대화 맥락을 기억하고, 사용자 질문에 점진적으로 더 나은 답을 제공.
- 문서 중심 질의응답: 벡터 저장소를 이

## 3. ReAct 패턴 이해하기
#####  ReAct(추론(reasoning)+행동(acting))는 프린스턴 대학의 야오(Yao)등이 2022년 발표한 프롬프트 엔지니어링 패러다임
##### stock_info_streamlit.py 와 같은 과정이 불필요

### 3.2 ReAct 과정 관찰하기
#### YAPF 패키지 설치



In [51]:
# %pip install yapf

#### 스타일 설정

- 기본값

```markdown
[style]
based_on_style = pep8
spaces_before_comment = 4
split_before_logical_operator = true
```

In [52]:
from yapf.yapflib.yapf_api import FormatCode
style = {'COLUMN_LIMIT': 50}

#### stream 메서드

In [53]:
query = "미국은 어떤 온도 단위를 주로 사용하나요?"
messages = [{
    "role": "user",
    "content": query
}]
for chunk in agent.stream(
    {"messages": messages},
    stream_mode="updates", # 에이전트 진행 상황 단위 스트림
):
    text, flag = FormatCode(str(chunk), style_config=style)
    print(text)

{
    'model': {
        'messages': [
            AIMessage(
                content=
                '미국은 주로 화씨(Fahrenheit) 단위를 사용합니다.\n\n- 일상 생활과 날씨 예보 등 대부분의 상황에서 화씨가 표기로 쓰입니다.\n- 섭씨(Celsius)는 과학, 의학, 국제 협력, 일부 교육 자료 등에서 널리 사용되지만, 일반적 생활 단위로는 화씨가 주를 이룹니다.\n- 켈빈(Kelvin)은 주로 과학 연구나 고급 이론 분야에서 사용됩니다.\n\n참고 예시:\n- 32°F = 0°C (물의 어는 점)\n- 212°F = 100°C (물의 끓는 점, 해수면에서)\n\n원하시면 특정 도시의 실시간 온도 단위 사용 여부나 예시를 더 자세히 알려드릴게요.',
                additional_kwargs={
                    'refusal': None
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 1014,
                        'prompt_tokens': 237,
                        'total_tokens': 1251,
                        'completion_tokens_details':
                        {
                            'accepted_prediction_tokens':
                            0,
                            'audio_tokens':
                            0,
                           

#### stream 메서드 출력 정리 
#### ReAct 패턴 출력

In [54]:
query = "5+2를 계산하고, 미국은 어떤 온도 단위를 주로 사용는지도 알려주세요."
messages = [{
        "role": "user",
        "content": query
    }]

for chunk in agent.stream(
    {"messages": messages},
    stream_mode="values",  # 에이전트 진행 상황 단위 스트림
):
    if "messages" not in chunk:
        continue

    latest_msg = chunk["messages"][-1]

    if latest_msg.__class__.__name__ == "AIMessage":
        # 도구 호출이 있으면 = 사고 과정 (Reasoning)
        if latest_msg.response_metadata.get('finish_reason') == 'tool_calls':
            # 도구 호출 중
            if latest_msg.tool_calls:
                for tc in latest_msg.tool_calls:
                    print(f"[Reasoning] 도구 호출: {tc['name']}")
                    print(f"    [Acting] {tc['name']}({tc['args']})")

        # 도구 호출이 없고 content가 있으면 = 최종 답변 (Finish)
        elif latest_msg.content:
            print(f"[Finish] {latest_msg.content}")

    elif latest_msg.__class__.__name__ == "ToolMessage": 
        print(f"[Observation] {latest_msg.content}")

[Reasoning] 도구 호출: math_calculator
    [Acting] math_calculator({'expression': '5+2'})
[Reasoning] 도구 호출: get_weather_units
    [Acting] get_weather_units({'city': '미국', 'units': '화씨'})
[Observation] 미국(은/는) 주로 화씨 단위를 사용합니다.
[Finish] - 5 + 2의 결과: 7
- 미국은 주로 화씨(화씨 단위, °F)를 사용합니다. 단, 과학이나 일부 맥락에서는 섭씨(°C)도 함께 사용되기도 합니다.


## 4. 외부 도구 사용하기
### 4.1 YouTubeSearchTool
#### 패키지 설치

In [62]:
%pip install -U youtube_search

Note: you may need to restart the kernel to use updated packages.


#### 도구 준비

In [63]:
from langchain_community.tools import YouTubeSearchTool

youtube_tool = YouTubeSearchTool()

#### 에이전트 준비

In [64]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5-nano", temperature=0)

media_system_prompt = (
    """
    당신은 한국어 미디어 큐레이터입니다.
    질문을 받으면 youtube_tool로 최신 영상 정보를 찾고,
    주제에 대해서 간략한 의견을 남겨주세요.
    """
 )

media_agent = create_agent(
    model=model,
    tools=[youtube_tool]    # youtube_tool 도구 탑재
)


#### 답변 생성

In [65]:
query = "와인 초보자를 위한 가이드 영상 찾아줘"
messages = [{"role": "user", "content": query}]

for chunk in media_agent.stream(
    {'messages': messages},
    stream_mode='updates'
):
    for step_name, state in chunk.items():
        print('step:', step_name)
        content = state['messages'][-1].content
        print('messages:', content)



step: model
messages: 
step: tools
messages: ['https://www.youtube.com/watch?v=pvi2qv5x34A&pp=ygUX7JmA7J24IOyeheusuCDqsIDsnbTrk5zSBwkJEwwBhyohjO8%3D']
step: tools
messages: ['https://www.youtube.com/watch?v=SJ86bgH4yfU&pp=ygUa7JmA7J24IOy0iOuztOyekCDqsIDsnbTrk5w%3D', 'https://www.youtube.com/watch?v=49N8_GKEV3g&pp=ygUa7JmA7J24IOy0iOuztOyekCDqsIDsnbTrk5w%3D']
step: model
messages: 다음은 와인 초보자를 위한 가이드 영상 후보들입니다. 링크를 열어 확인해 보세요. 원하시면 영상 제목이나 간단 요약도 함께 정리해 드리겠습니다.

- https://www.youtube.com/watch?v=pvi2qv5x34A&pp=ygUX7JmA7J24IOyeheusuCDqsIDsnbTrk5zSBwkJEwwBhyohjO8%3D
- https://www.youtube.com/watch?v=SJ86bgH4yfU&pp=ygUa7JmA7J24IOy0iOuztOyekCDqsIDsnbTrk5w%3D
- https://www.youtube.com/watch?v=49N8_GKEV3g&pp=ygUa7JmA7J24IOy0iOuztOyekCDqsIDsnbTrk5w%3D

추가로 원하시는 형식은 다음 중 어떤가요?
- 영상 제목/간단 요약 제공
- 한국어 자막 여부 확인 및 한국어 영상 우선 추천
- 더 많은 초보자용 영상 검색 (필터링: 난이도 낮음, 길이 짧음 등)
